# Stable-Baselines3 DQN으로 CartPole 테스트

위에서부터 순서대로 실행하세요. CartPole 화면을 84×84 흑백 이미지로 변환하고 최근 4장을 쌓아 입력하는 `CnnPolicy`를 학습합니다.
학습 후 별도 환경에서 10개 에피소드를 평가하고, 플레이를 애니메이션으로 확인합니다.

참고: [SB3 DQN 공식 문서](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html)


In [ ]:
%pip install stable-baselines3 "gymnasium[classic-control]>=1.0" matplotlib opencv-python


In [7]:
import gymnasium as gym
import matplotlib.pyplot as plt
from gymnasium.wrappers import (
    AddRenderObservation,
    ResizeObservation,
    GrayscaleObservation,
    FrameStackObservation,
)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

ENV_ID = "CartPole-v1"
SEED = 42
TOTAL_TIMESTEPS = 100_000


def make_pixel_env():
    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = AddRenderObservation(env, render_only=True)
    env = ResizeObservation(env, (84, 84))
    env = GrayscaleObservation(env, keep_dim=False)
    env = FrameStackObservation(env, stack_size=4)
    # (channels, height, width) = (4, 84, 84), uint8 [0, 255].
    # CnnPolicy가 내부적으로 /255 정규화를 수행합니다.
    return Monitor(env)


## 학습

MPS에서 학습합니다. MPS를 지원하지 않는 환경에서는 `device="cpu"`로 변경하세요.

이미지 관측의 replay buffer는 10,000개로 설정합니다. 현재/다음 관측 배열에 약 565 MB가 필요하며 모델과 학습 배치 메모리는 별도입니다. `TOTAL_TIMESTEPS`로 학습량을 조절할 수 있으며, 점수는 학습량과 시드에 따라 달라집니다.


In [8]:
train_env = make_pixel_env()
try:
    model = DQN(
        "CnnPolicy",
        train_env,
        learning_rate=1e-3,
        buffer_size=10_000,
        learning_starts=1_000,
        batch_size=64,
        gamma=0.99,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=500,
        exploration_fraction=0.2,
        exploration_final_eps=0.05,
        policy_kwargs=dict(net_arch=[64, 64]),
        seed=SEED,
        device="mps",
        verbose=1,
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS, log_interval=20)
finally:
    train_env.close()


Using mps device
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25.1     |
|    ep_rew_mean      | 25.1     |
|    exploration_rate | 0.976    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 379      |
|    time_elapsed     | 1        |
|    total_timesteps  | 501      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 23       |
|    ep_rew_mean      | 23       |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 40       |
|    fps              | 378      |
|    time_elapsed     | 2        |
|    total_timesteps  | 920      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21.7     |
|    ep_rew_mean      | 21.7     |
|    exploration_rate | 0.938    |
| t

## 평가

탐험 없이 학습한 정책으로 10회 평가합니다. CartPole-v1의 에피소드 최대 점수는 500입니다.


In [ ]:
eval_env = make_pixel_env()
try:
    eval_env.reset(seed=SEED + 1)
    mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=100,
        deterministic=True,
    )
    print(f"평균 보상: {mean_reward:.1f} ± {std_reward:.1f} / 500")
finally:
    eval_env.close()


평균 보상: 45.9 ± 16.1 / 500


## 플레이 확인

한 에피소드를 노트북 안에서 재생합니다.


In [10]:
render_env = make_pixel_env()
frames = []
episode_reward = 0.0
try:
    obs, info = render_env.reset(seed=SEED + 2)
    frames.append(render_env.render())
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(int(action))
        episode_reward += reward
        frames.append(render_env.render())
        if terminated or truncated:
            break
finally:
    render_env.close()

print(f"플레이 보상: {episode_reward:.0f}")
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(frames[0])
ax.axis("off")

def update(frame):
    image.set_data(frame)
    return (image,)

# 매 두 프레임을 표시해 HTML 크기를 줄입니다 (원래 재생 속도 유지).
animation = FuncAnimation(fig, update, frames=frames[::2], interval=40)
plt.close(fig)
display(HTML(animation.to_jshtml()))


플레이 보상: 64
